# Cobblemon Atlas｜基礎姿態＋疊加動作試渲染

專門測試超夢的 `standing + blink`，以及普通烈空座的 `standing + cry / physical_attack / special_attack`。每一個動作都有同時間軸的基礎姿態對照，只有真的增加可見畫面變化才輸出候選動畫。

**不修改正式網站、既有 1,170 張 WebP、Drive `renders/` 或 checkpoint。** 所有結果放在 `reports/resolver-validation/layered-action-pilot/`。


In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,time
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT=Path('/content/cobblemon-bedrock')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
print('Layered overlay pilot: Mewtwo + Rayquaza')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
tool=SITE_ROOT/'tools/render_layered_pilot.py'
index=DRIVE_ROOT/'reports/resolver-validation/resolved-forms.json'
assert tool.is_file() and index.is_file(), '缺少渲染程式或 Resolver 解析報告'
out=DRIVE_ROOT/'reports/resolver-validation/layered-action-pilot'
cmd=[sys.executable,str(tool),'--repo',str(WORK_ROOT),
     '--resolved',str(index),'--out',str(out)]
run=subprocess.run(cmd,check=False)
assert (out/'layered-render-pilot.json').is_file(), '未產生報告，請檢查 Colab 上方錯誤'
report=json.loads((out/'layered-render-pilot.json').read_text())
for form,entry in report['forms'].items():
    print('\nFORM',form,entry['status'],entry.get('reason'))
    for name,trial in entry.get('trials',{}).items():
        print(' ',name,trial['status'],'overlay extra pixels:',
              trial.get('overlay_change_pixels_max'), 'reason:',trial.get('reason'))
print('\nReport:',out/'layered-render-pilot.json')
print('Exit code:',run.returncode)
